<a href="https://colab.research.google.com/github/soralh1611/vertex-ai/blob/main/LLM_Powered_Loan_Management_System_w_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.cloud import aiplatform

# Define your project variables
PROJECT_ID = "soral-vertex-a"
REGION = "us-central1"
BUCKET_URI = "gs://soral-lms_bucket" # Used for storing model artifacts

# Initialize the Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

print(f"Vertex AI SDK initialized for project {PROJECT_ID}")

Vertex AI SDK initialized for project soral-vertex-a


In [2]:
from google.colab import auth
auth.authenticate_user()

import vertexai
vertexai.init(project="soral-vertex-a", location="us-central1")

In [3]:
from google.cloud import storage

def create_new_bucket(bucket_name, location="us-central1"):
    """Creates a new bucket in the specified location."""
    storage_client = storage.Client(project=PROJECT_ID)

    # 1. Clean name (remove gs:// if accidentally added)
    clean_name = bucket_name.replace("gs://", "").lower()

    try:
        # 2. Check if it already exists
        if storage_client.lookup_bucket(clean_name):
            print(f"⚠️ Bucket '{clean_name}' already exists.")
            return storage_client.get_bucket(clean_name)

        # 3. Create the bucket
        bucket = storage_client.create_bucket(clean_name, location=location)

        # 4. Optional: Enable Uniform Bucket-Level Access (Recommended for AI projects)
        bucket.iam_configuration.uniform_bucket_level_access_enabled = True
        bucket.patch()

        print(f"✅ Success: Bucket '{bucket.name}' created in {location}")
        return bucket

    except Exception as e:
        print(f"❌ Error creating bucket: {e}")

# Call the function with a unique name
# Tip: Use your name or project ID as a prefix
NEW_BUCKET_NAME = "lms-reports-soral-2025"
my_bucket = create_new_bucket(NEW_BUCKET_NAME)

⚠️ Bucket 'lms-reports-soral-2025' already exists.


In [4]:
pip install faker reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.6 MB/s eta 0:00:00


In [5]:
from faker import Faker
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import random

fake = Faker()

def generate_bank_statement(filename, account_holder):
    doc = SimpleDocTemplate(filename)
    elements = []
    styles = getSampleStyleSheet()

    # Header
    elements.append(Paragraph(f"<b>Bank of Vertex AI - Monthly Statement</b>", styles['Title']))
    elements.append(Paragraph(f"Account Holder: {account_holder}", styles['Normal']))
    elements.append(Paragraph(f"Statement Period: Dec 2025", styles['Normal']))

    # Transaction Data
    data = [["Date", "Description", "Amount", "Balance"]]
    balance = 5000.00
    for _ in range(15):
        date = f"2025-12-{random.randint(1, 20):02d}"
        desc = fake.company()
        amt = round(random.uniform(-500, 1000), 2)
        balance += amt
        data.append([date, desc, f"${amt}", f"${round(balance, 2)}"])

    # Table Styling
    t = Table(data)
    t.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(t)
    doc.build(elements)

generate_bank_statement("bank_statement_demo.pdf", "John Doe")

In [6]:
pip install faker faker-credit-score reportlab

In [7]:
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from faker_credit_score import CreditScore
from faker.providers import DynamicProvider
from faker.providers import BaseProvider

fake = Faker()
fake.add_provider(CreditScore)

def generate_credit_report(filename, applicant_name):
    doc = SimpleDocTemplate(filename, pagesize=LETTER)
    styles = getSampleStyleSheet()
    elements = []

    # Custom Style for "Confidential" Header
    header_style = ParagraphStyle('HeaderStyle', parent=styles['Normal'], fontSize=10, textColor=colors.red)

    # 1. Header Section
    elements.append(Paragraph("EQUIFAX - CONFIDENTIAL CONSUMER CREDIT FILE", header_style))
    elements.append(Spacer(1, 12))
    elements.append(Paragraph(f"<b>Subject:</b> {applicant_name}", styles['Title']))
    elements.append(Paragraph(f"<b>File Number:</b> {fake.uuid4()}", styles['Normal']))
    elements.append(Paragraph(f"<b>Date of Report:</b> Dec 21, 2025", styles['Normal']))
    elements.append(Spacer(1, 20))

    # 2. Credit Score Section (The "Big Number")
    score = fake.credit_score()
    score_data = [[f"EQUIFAX BEACON 5.0 SCORE: {score}"]]
    score_table = Table(score_data, colWidths=[400])
    score_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.lightgrey),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTSIZE', (0, 0), (-1, -1), 18),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 10),
    ]))
    elements.append(score_table)
    elements.append(Spacer(1, 20))

    # 3. Trade Lines (Credit Accounts)
    elements.append(Paragraph("<b>ACCOUNT HISTORY (TRADE LINES)</b>", styles['Heading2']))
    trade_data = [["Creditor", "Account Type", "Balance", "Status"]]

    # 1. DEFINE the class first
    class BankProvider(BaseProvider):
      def bank_name(self):
          banks = [
              "Chase Bank", "Wells Fargo", "Bank of America",
              "Vertex AI Financial", "Gemini Trust", "Goldman Sachs",
              "PNC Bank", "Citigroup", "Barclays"
          ]
          return self.random_element(banks)
    # 4. Add your custom provider to the Faker instance
    fake.add_provider(BankProvider)
    for _ in range(5):
        trade_data.append([
            fake.bank_name(),
            random.choice(["Revolving", "Installment", "Mortgage"]),
            f"${fake.random_int(0, 15000)}",
            random.choice(["Current", "30 Days Past Due", "Paid as Agreed"])
        ])

    t = Table(trade_data, colWidths=[150, 100, 80, 120])
    t.setStyle(TableStyle([
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('BACKGROUND', (0, 0), (-1, 0), colors.whitesmoke)
    ]))
    elements.append(t)

    doc.build(elements)

    from google.cloud import storage


def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client()
    clean_name = bucket_name.replace("gs://", "")
    bucket = storage_client.get_bucket(clean_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"✅ Success: Uploaded {source_file_name} to {clean_name}")

# --- EXECUTION STEPS ---

# Set your names
MY_BUCKET = "lms-reports-soral-2025"
FILE_NAME = "synthetic_report.pdf"

# STEP 1: Generate the file (Fixes Errno 2)
generate_credit_report(FILE_NAME, "Alex Rivera")

# STEP 2: Now that the file exists, upload it (Fixes 404)
upload_to_gcs(MY_BUCKET, FILE_NAME, "reports/december_report_01.pdf")



✅ Success: Uploaded synthetic_report.pdf to lms-reports-soral-2025


In [8]:
import os
import random
from faker import Faker
from faker.providers import BaseProvider
from faker_credit_score import CreditScore
from google.cloud import storage
from google.cloud.storage import transfer_manager
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# 1. SETUP
fake = Faker()
class BankProvider(BaseProvider):
    def bank_name(self):
        return self.random_element(["Chase", "Wells Fargo", "Vertex AI Bank", "Gemini Trust"])

fake.add_provider(BankProvider)
fake.add_provider(CreditScore)

BUCKET_NAME = "lms-reports-soral-2025"
LOCAL_DIR = "bulk_data_reports"
os.makedirs(LOCAL_DIR, exist_ok=True)
styles = getSampleStyleSheet()

# 2. DATA GENERATION FUNCTION
def generate_full_report(i):
    name = fake.name()
    u_id = f"{i:04d}"
    filename = os.path.join(LOCAL_DIR, f"report_{u_id}.pdf")

    doc = SimpleDocTemplate(filename)

    # --- CRITICAL: Create a NEW story list for every file ---
    story = []

    # Add Title
    story.append(Paragraph(f"<b>Financial Audit: {name}</b>", styles['Title']))
    story.append(Spacer(1, 12))

    # Add Financial Summary
    summary_data = [
        ["Metric", "Value"],
        ["Credit Score", str(fake.credit_score())],
        ["Monthly Income", f"${random.randint(3000, 12000)}"],
        ["Primary Bank", fake.bank_name()]
    ]
    summary_table = Table(summary_data, colWidths=[150, 150])
    summary_table.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.lightgrey), ('GRID', (0,0), (-1,-1), 1, colors.black)]))
    story.append(summary_table)
    story.append(Spacer(1, 20))

    # Add 15 Mock Transactions (Ensures file is NOT empty)
    trans_data = [["Date", "Merchant", "Amount", "Type"]]
    for _ in range(15):
        trans_data.append([
            str(fake.date_this_year()),
            fake.company(),
            f"${random.randint(-1000, 2000)}",
            random.choice(["Debit", "Credit", "ACH"])
        ])

    trans_table = Table(trans_data, colWidths=[80, 150, 80, 80])
    trans_table.setStyle(TableStyle([('GRID', (0,0), (-1,-1), 0.5, colors.grey), ('FONTSIZE', (0,0), (-1,-1), 8)]))
    story.append(trans_table)

    # FINAL STEP: Build PDF
    doc.build(story)
    return filename

# 3. RUN & UPLOAD
def run_bulk_and_upload(count=1000):
    all_filenames = []
    print(f"🛠️ Generating {count} data-rich reports...")
    for i in range(count):
        all_filenames.append(os.path.basename(generate_full_report(i)))
        if i % 100 == 0: print(f"Progress: {i}/{count}")

    print("🚀 Bulk Uploading to GCS...")
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)

    transfer_manager.upload_many_from_filenames(
        bucket,
        all_filenames,
        source_directory=LOCAL_DIR,
        max_workers=8
    )
    print("✅ All 1,000 files uploaded with data.")

run_bulk_and_upload(1000)

🛠️ Generating 1000 data-rich reports...
Progress: 0/1000
Progress: 100/1000
Progress: 200/1000
Progress: 300/1000
Progress: 400/1000
Progress: 500/1000
Progress: 600/1000
Progress: 700/1000
Progress: 800/1000
Progress: 900/1000
🚀 Bulk Uploading to GCS...
✅ All 1,000 files uploaded with data.


In [9]:
pip install streamlit google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.4 MB/s eta 0:00:00


In [10]:
!pip install -q streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 6s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [13]:
!pip install -q gradio google-genai

In [27]:
import gradio as gr
from google import genai

# --- CONFIG ---
PROJECT_ID = "soral-vertex-a"
LOCATION = "us-central1"
MODEL_ID = "gemini-2.0-flash-001" # Your Gemini 2 family model

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

def fetch_report(customer_id):
    try:
        # Check if ID is just numbers
        clean_id = "".join(filter(str.isdigit, str(customer_id)))
        blob = bucket.blob(f"credit-reports/{clean_id}.json")

        if not blob.exists():
            print(f"⚠️ ID {clean_id} not found in GCS.")
            return None

        data = json.loads(blob.download_as_text())
        return data
    except Exception as e:
        print(f"❌ GCS Fetch Failed: {str(e)}")
        return None

# 1. SETUP THE TOOL (The standard 2025 Syntax)
# Ensure PROJECT_ID is 'soral-vertex-a'
# Ensure DATASTORE_ID is 'artha-connector_1766452201214_gcs_store'
DATA_STORE_PATH = f"projects/soral-vertex-a/locations/global/collections/default_collection/dataStores/artha-connector_1766452201214_gcs_store"

rag_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_ai_search=types.VertexAISearch(
            datastore=DATA_STORE_PATH
        )
    )
)


def loan_agent_chat(message, history):
    # message: the current user input
    # history: previous chat turns
    # Check if the input is a 3-digit ID
    customer_id = "".join(filter(str.isdigit, message))

    report_context = ""
    if len(customer_id) == 3:
        data = fetch_report(customer_id)
        if data:
            report_context = f"\nCUSTOMER DATA DETECTED: {json.dumps(data)}"
        else:
            report_context = "\n(System: Customer ID not found in database, a customer agent will call you.)"

    system_instruction = """
    You are Artha, a versatile AI assistant with a specialty in FinTech.

    GENERAL MODE: If the user asks general questions (history, science, jokes, etc.),
    answer as a helpful, creative LLM.

    FINANCE MODE: If the user asks about loans, credit policies, or their specific
    application/ID, you should check the RAG documents.

    STRICT RULES:
    - Never approve a loan unless the USER DATA matches the PDF rules.
    - Be playful and witty, but professionally firm on financial limits.
    """

    # 2. Call Gemini
    response = client.models.generate_content(
            model="gemini-2.5-pro",
            contents=message,
            config=types.GenerateContentConfig(
                tools=[rag_tool],
                # This ensures Artha is playful but uses the PDF
                system_instruction= system_instruction
            )
        )

    return response.text

# --- LAUNCH UI ---
# This creates a beautiful, working chat window directly in Colab
demo = gr.ChatInterface(
    fn=loan_agent_chat,
    title="Artha Loan Underwriter",
    description="Ask about your loan eligibility. Grounded in Gemini 2.5 Flash.",
    theme="soft"
)

# share=True creates a public URL that won't give you 'Failed to Fetch' errors
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4379fa93762187edb7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [24]:
from google.genai import types

# 1. SETUP THE TOOL (The standard 2025 Syntax)
# Ensure PROJECT_ID is 'soral-vertex-a'
# Ensure DATASTORE_ID is 'artha-connector_1766452201214_gcs_store'
DATA_STORE_PATH = f"projects/soral-vertex-a/locations/global/collections/default_collection/dataStores/artha-connector_1766452201214_gcs_store"

rag_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_ai_search=types.VertexAISearch(
            datastore=DATA_STORE_PATH
        )
    )
)

# 2. THE CHAT FUNCTION
def loan_agent_chat(user_query, history):
    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash-001",
            contents=user_query,
            config=types.GenerateContentConfig(
                tools=[rag_tool],
                # This ensures Loomis is playful but uses the PDF
                system_instruction="You are Loomis, the playful Artha underwriter. Use the credit policy PDF to decide."
            )
        )
        return response.text
    except Exception as e:
        # If there is a permission error, it will show here instead of crashing
        return f"Artha Vault Connection Error: {str(e)}"

In [127]:
import json
import os
from google.cloud import storage

# --- 1. CONFIG ---
BUCKET_NAME = "lms-reports-soral-2025"
FOLDER_NAME = "credit-reports"

# 2. GENERATE DATA
reports = []
# 10 PASSING (IDs 101-110)
for i in range(101, 111):
    reports.append({
        "id": str(i), "name": f"Qualified Customer {i}", "fico": 750,
        "annual_income": 100000, "requested_loan": 20000, # LTI 20%
        "inquiries_120d": 0, "foreclosures_24m": 0, "new_trades_24m": 0, "cc_utilization": 30
    })

# 10 FAILING (IDs 901-910) - Each fails one specific rule
scenarios = [
    {"id": "901", "requested_loan": 90000, "annual_income": 100000}, # Fail LTI (90%)
    {"id": "902", "inquiries_120d": 2},                              # Fail Inquiries
    {"id": "903", "foreclosures_24m": 1},                            # Fail Foreclosure
    {"id": "904", "new_trades_24m": 1},                               # Fail New Trade
    {"id": "905", "cc_utilization": 95},                             # Fail Utilization
    {"id": "906", "fico": 620},                                      # Fail Base FICO
    {"id": "907", "inquiries_120d": 1, "cc_utilization": 85},        # Fail Multiple
    {"id": "908", "requested_loan": 60000, "annual_income": 70000}, # Fail LTI (85%)
    {"id": "909", "new_trades_24m": 2},                               # Fail New Trade
    {"id": "910", "foreclosures_24m": 1, "fico": 800}                # Fail Foreclosure
]
reports.extend(scenarios)

# 3. UPLOAD TO GCS
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

for r in reports:
    filename = f"{r['id']}.json"
    blob = bucket.blob(f"{FOLDER_NAME}/{filename}")
    blob.upload_from_string(json.dumps(r, indent=2), content_type='application/json')

print(f"✅ Successfully uploaded 20 reports to gs://{BUCKET_NAME}/{FOLDER_NAME}/")

✅ Successfully uploaded 20 reports to gs://lms-reports-soral-2025/credit-reports/


In [23]:
def fetch_report(customer_id):
    """Downloads the JSON report from GCS based on ID"""
    try:
        blob = bucket.blob(f"credit-reports/{customer_id}.json")
        data = json.loads(blob.download_as_text())
        return data
    except:
        return None

def loan_agent_chat(user_input, history):
    # Check if the input is a 3-digit ID
    customer_id = "".join(filter(str.isdigit, user_input))

    report_context = ""
    if len(customer_id) == 3:
        data = fetch_report(customer_id)
        if data:
            report_context = f"\nCUSTOMER DATA DETECTED: {json.dumps(data)}"
        else:
            report_context = "\n(System: Customer ID not found in database, a customer agent will call you.)"

    # Call Gemini with RAG Tool and the Report Context
    try:
        response = client.models.generate_content(
            model="gemini-2.5-pro",
            contents=f"{user_input} {report_context}",
            config=types.GenerateContentConfig(
                tools=[rag_tool],
                system_instruction="You are Loomis. If a CUSTOMER DATA block is provided, evaluate it strictly against the Artha Credit Policy PDF."
            )
        )
        return response.text
    except Exception as e:
        return f"Error: {str(e)}"